# K-Means Clustering

In this module, we will use K-Means Clustering to segment
customers based on their purchasing behavior.

We will use a real-world Kaggle Online Retail dataset.

## Machine Learning Problem

Input:

Customer purchasing behavior

Features:

- Recency
- Frequency
- Monetary

Output:

Customer segments / clusters

The clusters will be discovered automatically by the
K-Means Clustering algorithm.

## Models

We will train:

1. K-Means Clustering

We will then investigate how the number of clusters
affects the clustering results.

## Workflow

Dataset
↓
Data Exploration
↓
Data Cleaning
↓
Customer-Level Data
↓
RFM Feature Engineering
↓
Feature Scaling
↓
K-Means Clustering
↓
Cluster Assignment
↓
Customer Segmentation

#Install Kaggle

In [ ]:
!pip install -q kaggle

#Import Libraries

In [ ]:
import os
from getpass import getpass

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

print("Libraries imported successfully.")

#Kaggle API Authentication

## Kaggle API Authentication

We will use `getpass()` to securely enter the Kaggle API token.

The token will not be displayed while entering it.

In [ ]:
kaggle_token = getpass("Enter your Kaggle API token: ")

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

del kaggle_token

print("Kaggle API token configured successfully.")

## Download the Online Retail Dataset

Kaggle Dataset:

`thedevastator/online-retail-sales-and-customer-data`

We will download and extract the dataset directly into
Google Colab.

In [ ]:
!mkdir -p /content/online_retail_dataset

!kaggle datasets download \
    -d thedevastator/online-retail-sales-and-customer-data \
    -p /content/online_retail_dataset \
    --unzip

#Check Downloaded Files

In [ ]:
os.listdir("/content/online_retail_dataset")

#Find Dataset Files

In [ ]:
dataset_files = [
    file
    for file in os.listdir("/content/online_retail_dataset")
    if file.lower().endswith((".csv", ".xlsx", ".xls"))
]

dataset_files

#Load Dataset

In [ ]:
dataset_path = os.path.join(
    "/content/online_retail_dataset",
    dataset_files[0]
)

if dataset_path.lower().endswith(".csv"):
    df = pd.read_csv(dataset_path)
else:
    df = pd.read_excel(dataset_path)

df.head()

#Dataset Shape

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

##Column Names

In [ ]:
print("Columns:")

for column in df.columns:
    print(column)

#Dataset Information

In [ ]:
df.info()

#Statistical Summary

In [ ]:
df.describe(include="all").T

## Missing Values

Before performing customer segmentation, we need to
identify missing values in the dataset.

Customer identification is especially important because
we will create customer-level features later.

In [ ]:
missing_values = df.isnull().sum()

missing_values[
    missing_values > 0
]

#Check Duplicate Rows

In [ ]:
print(
    "Duplicate rows:",
    df.duplicated().sum()
)

#Customer Identification

The customer identifier allows us to group individual
transactions belonging to the same customer.

We will use this identifier later to create customer-level
RFM features.

RFM represents:

R → Recency
F → Frequency
M → Monetary

In [ ]:
print(df.columns.tolist())

#Data Cleaning

Before creating customer segments, we need to clean the
transaction data.

We will:

- Remove transactions without a Customer ID
- Remove cancelled transactions
- Remove invalid quantities
- Remove invalid prices

In [ ]:
print("Original shape:", df.shape)

df = df.dropna(
    subset=["CustomerID"]
)

df = df[
    ~df["InvoiceNo"].astype(str).str.startswith("C")
]

df = df[
    df["Quantity"] > 0
]

df = df[
    df["UnitPrice"] > 0
]

print("Cleaned shape:", df.shape)

#Verify Cleaned Dataset

In [ ]:
print(
    "Missing Customer IDs:",
    df["CustomerID"].isnull().sum()
)

print(
    "Invalid Quantity:",
    (df["Quantity"] <= 0).sum()
)

print(
    "Invalid Unit Price:",
    (df["UnitPrice"] <= 0).sum()
)

#Create Total Amount

Each transaction contains:

Quantity

and

Unit Price

We can calculate the total transaction value as:

Total Amount = Quantity × Unit Price

This value will be used to calculate the Monetary
component of RFM.

In [ ]:
df["TotalAmount"] = (
    df["Quantity"] *
    df["UnitPrice"]
)

df[
    [
        "Quantity",
        "UnitPrice",
        "TotalAmount"
    ]
].head()

#Transaction Date

The InvoiceDate column represents when a transaction
occurred.

We will convert it into a proper datetime format so that
we can calculate customer recency.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"]
)

print(
    df["InvoiceDate"].min()
)

print(
    df["InvoiceDate"].max()
)

#RFM Customer Segmentation

We will transform transaction-level data into
customer-level data.

RFM represents three important customer behaviors:

### Recency

How recently did the customer purchase?

### Frequency

How frequently did the customer purchase?

### Monetary

How much money did the customer spend?

Each customer will therefore be represented by
three numerical features.

#Set Reference Date

To calculate Recency, we need a reference date.

We will use the day after the last transaction in the
dataset.

A customer who purchased recently will have a smaller
Recency value.

In [ ]:
reference_date = (
    df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

reference_date

#Create RFM Features

In [ ]:
rfm = df.groupby(
    "CustomerID"
).agg(
    Recency=(
        "InvoiceDate",
        lambda x: (
            reference_date - x.max()
        ).days
    ),
    Frequency=(
        "InvoiceNo",
        "nunique"
    ),
    Monetary=(
        "TotalAmount",
        "sum"
    )
)

rfm.head()

#Customer-Level Dataset

The original dataset contains individual transactions.

The RFM dataset contains one row per customer.

Therefore:

Transaction Data
↓
Group by Customer
↓
RFM Features
↓
Customer-Level Dataset

In [ ]:
print(
    "Number of customers:",
    rfm.shape[0]
)

print(
    "Number of features:",
    rfm.shape[1]
)

rfm.head()

#RFM Statistical Summary

Before applying K-Means, we will examine the
distribution of the three customer-level features.

This helps us understand the scale and spread of:

- Recency
- Frequency
- Monetary

In [ ]:
rfm.describe().T

#RFM Feature Distributions

In [ ]:
rfm.hist(
    figsize=(14, 5),
    bins=30
)

plt.suptitle(
    "RFM Feature Distributions"
)

plt.show()

#RFM Feature Correlation

Let's examine the relationships between:

- Recency
- Frequency
- Monetary

This helps us understand the customer behavior
before clustering.

In [ ]:
plt.figure(
    figsize=(8, 6)
)

correlation_matrix = rfm.corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0
)

plt.title(
    "RFM Feature Correlation Matrix"
)

plt.show()

#Feature Scaling

K-Means clustering is based on distances between data
points.

If one feature has much larger values than another,
it can dominate the distance calculation.

For example:

Recency → 1 to 300

Frequency → 1 to 100

Monetary → 10 to 10000

Monetary could have a disproportionately large influence.

Therefore, we will standardize the RFM features.

We will use StandardScaler:

z = (x - mean) / standard deviation

#Create StandardScaler

In [ ]:
scaler = StandardScaler()

#Scale RFM Features

In [ ]:
rfm_scaled = scaler.fit_transform(
    rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
)

print(
    "RFM features scaled successfully."
)

#Verify Scaling

In [ ]:
scaled_rfm_df = pd.DataFrame(
    rfm_scaled,
    columns=[
        "Recency",
        "Frequency",
        "Monetary"
    ]
)

print("Mean:")

display(
    scaled_rfm_df.mean().round(2)
)

print("\nStandard Deviation:")

display(
    scaled_rfm_df.std().round(2)
)

#K-Means Clustering

K-Means is an unsupervised learning algorithm.

Unlike supervised learning:

There is no target variable.

The algorithm tries to divide the observations into
K groups called clusters.

The basic process is:

1. Choose K clusters.
2. Initialize K centroids.
3. Assign each observation to the nearest centroid.
4. Recalculate the centroids.
5. Repeat the process until the centroids stabilize.

The objective is to minimize the distance between
observations and their assigned cluster centers.

#Choose Initial Number of Clusters

For our initial K-Means model, we will use:

K = 4

This is only an initial value.

Later, we will investigate different values of K
using methods such as:

- Elbow Method
- Silhouette Score

In [ ]:
k = 4

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

kmeans.fit(
    rfm_scaled
)

print(
    "K-Means model trained successfully."
)

#Cluster Labels

After training, K-Means assigns every customer to
one of the K clusters.

The cluster labels are represented as:

0
1
2
...
K-1

In [ ]:
cluster_labels = kmeans.labels_

print(
    cluster_labels[:20]
)

#Add Cluster Labels

In [ ]:
rfm["Cluster"] = cluster_labels

rfm.head()

#Cluster Distribution

Let's examine how many customers have been assigned
to each cluster.

This gives us an initial understanding of the
customer segmentation.

In [ ]:
cluster_counts = (
    rfm["Cluster"]
    .value_counts()
    .sort_index()
)

cluster_counts

#Visualize Cluster Distribution

In [ ]:
plt.figure(
    figsize=(8, 5)
)

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.title(
    "Customer Distribution Across Clusters"
)

plt.show()

#Cluster Centers

Each K-Means cluster has a centroid.

The centroid represents the average position of
the customers belonging to that cluster in the
scaled feature space.

We will inspect the centroid values to understand
the structure of the clusters.

In [ ]:
cluster_centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=[
        "Recency",
        "Frequency",
        "Monetary"
    ]
)

cluster_centers

#Cluster Profiles

We will now calculate the average RFM values for
each customer cluster.

This allows us to interpret what each cluster
represents.

In [ ]:
cluster_profile = (
    rfm.groupby("Cluster")[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]
    .mean()
    .round(2)
)

cluster_profile

#Part 1 Complete

We have now:

- Loaded a real-world retail dataset.
- Explored the transaction data.
- Checked missing values.
- Checked duplicate rows.
- Cleaned invalid transactions.
- Created Total Amount.
- Converted transaction dates.
- Created customer-level RFM features.
- Explored RFM distributions.
- Examined feature correlations.
- Standardized the RFM features.
- Trained an initial K-Means model.
- Assigned customers to clusters.
- Examined cluster distribution.
- Examined cluster centers.
- Created initial customer cluster profiles.

## Current Workflow

Retail Transactions
↓
Data Exploration
↓
Data Cleaning
↓
Total Amount
↓
RFM Features
↓
Feature Scaling
↓
K-Means
↓
Cluster Assignment
↓
Cluster Profiles

In Part 2, we will investigate how to select the
appropriate number of clusters and evaluate the
quality of the clustering.

# Part 2 — K-Means Evaluation, Cluster Selection & Visualization

In Part 1, we prepared customer-level RFM features, scaled the data, and trained an initial K-Means model using 4 clusters.

In Part 2, we will:

1. Determine the appropriate number of clusters
2. Use the Elbow Method
3. Use Silhouette Score
4. Use Davies-Bouldin Index
5. Compare different values of K
6. Train the final K-Means model
7. Analyze the final customer segments
8. Visualize the clusters
9. Interpret the business meaning of each cluster

## Why Do We Need to Choose K?

K-Means requires us to specify the number of clusters (`K`) before training the model.

For example:

- K = 2 → 2 customer segments
- K = 3 → 3 customer segments
- K = 4 → 4 customer segments
- K = 5 → 5 customer segments

Choosing a very small or very large value of K may produce clusters that are not useful.

Therefore, we need evaluation techniques to help us select a suitable value of K.

## 1. Elbow Method

The Elbow Method uses **inertia** to evaluate different values of K.

Inertia measures the total squared distance between each data point and the centroid of its assigned cluster.

Lower inertia means that the points are closer to their cluster centers.

However, increasing K will almost always decrease inertia.

We therefore look for an **"elbow"** — a point after which increasing K provides only a small improvement.

### Interpretation

- Lower inertia → tighter clusters
- Large decrease → significant improvement
- Small decrease → limited improvement
- Elbow → potential suitable value of K

In [ ]:
inertia_values = []

k_values = range(2, 11)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(rfm_scaled)
    inertia_values.append(model.inertia_)

print("Inertia values calculated successfully.")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(k_values, inertia_values, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")

plt.xticks(k_values)
plt.grid(True)

plt.show()

### Understanding the Elbow Plot

The curve generally decreases as K increases.

At first, increasing K significantly reduces inertia.

After a certain point, the improvement becomes much smaller.

This point is called the **elbow**.

The elbow provides a candidate value for the number of clusters.

However, we should not rely on the Elbow Method alone.

We will also use:

- Silhouette Score
- Davies-Bouldin Index

### Understanding the Elbow Plot

The curve generally decreases as K increases.

At first, increasing K significantly reduces inertia.

After a certain point, the improvement becomes much smaller.

This point is called the **elbow**.

The elbow provides a candidate value for the number of clusters.

However, we should not rely on the Elbow Method alone.

We will also use:

- Silhouette Score
- Davies-Bouldin Index

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_values = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(rfm_scaled)

    score = silhouette_score(rfm_scaled, labels)
    silhouette_values.append(score)

print("Silhouette scores calculated successfully.")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    silhouette_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(k_values)
plt.grid(True)

plt.show()

### Understanding the Silhouette Plot

We prefer a value of K that produces a relatively high Silhouette Score.

A high score indicates that:

- Customers within the same cluster are similar.
- Customers in different clusters are relatively different.
- The clusters are reasonably well separated.

The highest score is useful as a candidate, but the final K should also make business sense.

## 3. Davies-Bouldin Index

The Davies-Bouldin Index evaluates the similarity between clusters.

It considers:

- How compact each cluster is
- How far apart different clusters are

For this metric:

> **Lower values are better.**

A lower Davies-Bouldin Index indicates that clusters are compact and well separated.

In [ ]:
from sklearn.metrics import davies_bouldin_score

davies_bouldin_values = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(rfm_scaled)

    score = davies_bouldin_score(rfm_scaled, labels)
    davies_bouldin_values.append(score)

print("Davies-Bouldin scores calculated successfully.")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    davies_bouldin_values,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Davies-Bouldin Index")
plt.title("Davies-Bouldin Index for Different K Values")

plt.xticks(k_values)
plt.grid(True)

plt.show()

## 4. Comparing Different Values of K

We now have three different evaluation measures:

| Metric | Preferred Result |
|---|---|
| Inertia | Lower, while looking for the elbow |
| Silhouette Score | Higher |
| Davies-Bouldin Index | Lower |

Instead of looking at only one metric, we can compare all of them together.

In [ ]:
evaluation_results = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertia_values,
    "Silhouette Score": silhouette_values,
    "Davies-Bouldin Index": davies_bouldin_values
})

evaluation_results

In [ ]:
evaluation_results.sort_values(
    by="Silhouette Score",
    ascending=False
)

## Selecting the Final K

The final value of K should be selected by considering:

1. The elbow in the inertia curve
2. A high Silhouette Score
3. A low Davies-Bouldin Index
4. The size of each cluster
5. Whether the resulting customer segments are meaningful

We should avoid selecting K purely because one metric has the best numerical value.

For customer segmentation, the clusters should also be interpretable from a business perspective.

## 5. Training the Final K-Means Model

After comparing the evaluation metrics, select the appropriate value of K.

The value below should be changed according to the evaluation results obtained above.

In [ ]:
final_k = 4

In [ ]:
final_kmeans = KMeans(
    n_clusters=final_k,
    random_state=42,
    n_init=10
)

final_labels = final_kmeans.fit_predict(rfm_scaled)

print("Final K-Means model trained successfully.")

In [ ]:
rfm["Cluster"] = final_labels

rfm.head()

## 6. Final Cluster Distribution

Let's examine how many customers belong to each cluster.

In [ ]:
final_cluster_counts = (
    rfm["Cluster"]
    .value_counts()
    .sort_index()
)

final_cluster_counts

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=final_cluster_counts.index,
    y=final_cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Customers")
plt.title("Final Customer Distribution by Cluster")

plt.show()

## 7. Customer Segment Profiling

The cluster number itself does not have a business meaning.

For example:

- Cluster 0 is not automatically the "best" customer group.
- Cluster 1 is not automatically the "worst" group.

We need to examine the RFM values of each cluster.

We will compare:

- Recency
- Frequency
- Monetary

In [ ]:
final_cluster_profile = (
    rfm.groupby("Cluster")[
        ["Recency", "Frequency", "Monetary"]
    ]
    .mean()
    .round(2)
)

final_cluster_profile

## Understanding the RFM Values

### Recency

Lower Recency is generally better.

A customer who purchased recently will have a smaller Recency value.

### Frequency

Higher Frequency generally indicates that the customer purchases more often.

### Monetary

Higher Monetary value indicates that the customer has spent more money.

Therefore, we can use these three characteristics to understand the customer segments.

In [ ]:
final_cluster_profile["Customer Count"] = final_cluster_counts

final_cluster_profile

## 8. Visualizing Customer Segments

RFM contains three dimensions:

- Recency
- Frequency
- Monetary

We can visualize pairs of these features to understand how the clusters are distributed.

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Frequency",
    hue="Cluster",
    palette="tab10"
)

plt.title("Customer Clusters: Recency vs Frequency")
plt.xlabel("Recency")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Monetary",
    hue="Cluster",
    palette="tab10"
)

plt.title("Customer Clusters: Recency vs Monetary")
plt.xlabel("Recency")
plt.ylabel("Monetary")

plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Cluster",
    palette="tab10"
)

plt.title("Customer Clusters: Frequency vs Monetary")
plt.xlabel("Frequency")
plt.ylabel("Monetary")

plt.show()

## 9. PCA Visualization

RFM contains three dimensions, so it is difficult to visualize all three dimensions simultaneously.

We can use **Principal Component Analysis (PCA)** to reduce the three-dimensional RFM data into two dimensions.

PCA is being used here only for visualization.

The K-Means model was trained using the original scaled RFM features.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

rfm_pca = pca.fit_transform(rfm_scaled)

print("PCA transformation completed.")

In [ ]:
pca_df = pd.DataFrame(
    rfm_pca,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = final_labels

pca_df.head()

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="tab10"
)

plt.title("Customer Segments Using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.show()

## 10. Interpreting the Customer Segments

We can now interpret each cluster based on its RFM profile.

A typical interpretation may follow this logic:

### High-Value Customers

- Low Recency
- High Frequency
- High Monetary

These customers purchase recently, purchase frequently, and spend more.

### Loyal Customers

- Relatively low Recency
- High Frequency
- Moderate or high Monetary

These customers regularly interact with the business.

### At-Risk Customers

- High Recency
- Previously high Frequency and/or Monetary

These customers have valuable purchasing histories but have not purchased recently.

### Low-Value / Inactive Customers

- High Recency
- Low Frequency
- Low Monetary

These customers have relatively low engagement and spending.

The actual segment names should be assigned after examining the final cluster profile.

In [ ]:
final_cluster_profile.sort_values(
    by=["Monetary", "Frequency"],
    ascending=False
)

## Business Interpretation

The purpose of customer segmentation is not simply to create clusters.

The goal is to use these clusters to make better business decisions.

For example:

| Customer Type | Possible Strategy |
|---|---|
| High Value | Loyalty rewards and exclusive offers |
| Loyal | Membership and retention programs |
| At Risk | Re-engagement campaigns |
| Low Value | Low-cost promotional campaigns |

The exact strategy should depend on the characteristics of the final clusters.

# Final K-Means Model Summary

The final model:

- Uses customer-level RFM features
- Uses standardized features
- Uses the selected number of clusters
- Assigns every customer to a cluster
- Produces customer segment profiles
- Can be used to support targeted marketing strategies

In [ ]:
print("Final Number of Clusters:", final_k)
print("Number of Customers:", len(rfm))
print("Number of Features:", 3)

print("\nRFM Features:")
print(["Recency", "Frequency", "Monetary"])

print("\nCluster Distribution:")
print(final_cluster_counts)

# K-Means Module Complete

## What We Covered

### Data Preparation
- Loaded the retail transaction dataset
- Explored the dataset
- Handled missing Customer IDs
- Removed cancelled transactions
- Removed invalid quantities and prices

### Feature Engineering
- Created Total Amount
- Converted InvoiceDate
- Created customer-level RFM features:
  - Recency
  - Frequency
  - Monetary

### Data Preprocessing
- Standardized the RFM features using StandardScaler

### K-Means Clustering
- Trained K-Means
- Assigned customers to clusters
- Examined cluster sizes
- Calculated cluster centers

### Model Evaluation
- Elbow Method
- Inertia
- Silhouette Score
- Davies-Bouldin Index

### Visualization
- Recency vs Frequency
- Recency vs Monetary
- Frequency vs Monetary
- PCA-based visualization

### Business Interpretation
- Analyzed customer behavior
- Compared customer segments
- Identified potential high-value, loyal, at-risk, and low-value groups

---

## Complete Workflow

**Retail Transactions**

↓

**Data Exploration**

↓

**Data Cleaning**

↓

**Total Amount**

↓

**RFM Feature Engineering**

↓

**Feature Scaling**

↓

**K-Means Clustering**

↓

**Elbow Method**

↓

**Silhouette Score**

↓

**Davies-Bouldin Index**

↓

**Select Optimal K**

↓

**Final K-Means Model**

↓

**Customer Clusters**

↓

**Cluster Profiling**

↓

**Visualization**

↓

**Business Interpretation**

---

## Key Concepts Learned

- Unsupervised Learning
- K-Means Clustering
- Centroids
- Euclidean Distance
- Inertia
- Elbow Method
- Silhouette Score
- Davies-Bouldin Index
- Feature Scaling
- RFM Analysis
- Customer Segmentation
- PCA Visualization